In [51]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import nest_asyncio
from nemoguardrails import RailsConfig ,LLMRails
from nemoguardrails.actions import action
from typing import Optional
import re

# from .autonotebook import tqdm as notebook_tqdm
nest_asyncio.apply()
load_dotenv()

True

In [52]:
MODEL = ChatGroq(model = 'openai/gpt-oss-120b')
GAURD_MODEL = ChatGroq(model = 'openai/gpt-oss-20b')

In [53]:
@action(is_system_action=True)
async def detect_PII(context: Optional[dict]= None):
    user_message = context.get("user_message" , "") if context else ""
    patterns = {
    "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    "phone": r"\b(\+\d{1,2}\s?)?(\(?\d{3}\)?[\s.-]?)?\d{3}[\s.-]?\d{4}\b|\b\d{11,15}\b",
    "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
    "api_key": r"(api[_\s-]?key|token|secret)[\:\s]+[A-Za-z0-9_\-]{16,}",
    "credit_card": r"\b\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}\b",
}
    found = [type for type,pat in patterns.items()
             if (re.search(pat , user_message , re.IGNORECASE))]
    return found



In [54]:

@action(is_system_action=True)
async def classify_urgency(context: Optional[dict] = None):
    """Returns True if the message signals a production emergency."""
    msg = (context.get("user_message", "") if context else "").lower()
    urgent_keywords = [
        "outage",
        "down",
        "crash",
        "critical",
        "emergency",
        "not working",
        "urgent",
        "p0",
        "p1",
    ]
    return any(kw in msg for kw in urgent_keywords)

In [55]:
Colang_action = """ 


define bot acknowledge the urgency
    "I will handle as soon as possible"

define bot ask to remove PII
    "Sorry, I found personal information in your query. Please remove it first."

define flow check input PII
    $pii_found = execute detect_PII
    if $pii_found
        bot ask to remove PII
        stop


define flow check input Urgency
    $urgency_found = execute classify_urgency
    if $urgency_found
        bot acknowledge the urgency


"""

In [56]:
yaml_content = """
models:
  - type: main
    engine: groq
    model: openai/gpt-oss-20b

instructions:
  - type: general
    content: |
      You are a smart AI assistant who can answer sports-related queries.
      Only answer questions related to sports.

rails:
  input:
    flows:
      - check input PII
      - check input Urgency
            
"""

In [57]:
config_exp5 = RailsConfig.from_content(
    colang_content = Colang_action,
    yaml_content = yaml_content
)
rail_exp5 = LLMRails(config_exp5 , GAURD_MODEL)

C:\Users\soodm\AppData\Local\Temp\ipykernel_2892\2494185119.py:5: DeprecationWarning: Passing a raw LangChain LLM is deprecated. Use LangChainLLMAdapter(llm) explicitly or pass an LLMModel instance.
  rail_exp5 = LLMRails(config_exp5 , GAURD_MODEL)


In [58]:
rail_exp5.register_action(detect_PII) 
rail_exp5.register_action(classify_urgency) 

In [59]:
def chat(rail, message):
    print('___'*62)
    response = rail.generate(messages=[{'role': 'user', 'content': message}])
    content = response.get("content", str(response))
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.IGNORECASE | re.DOTALL).strip()
    print(f"Bot Reply: {content}")
    print('___'*62)
    return content

In [64]:
attempts = [
    "hi, my phone number is 12345678901, can you check my tickets are booked for not?",
    "Ticket booking Services are down please help",
    "can you check any new and ticket for cricket match for user@example.com",
    "Can you tell me what is asain games ?"

]

for attempt in attempts:
    chat(rail_exp5 , attempt)

__________________________________________________________________________________________________________________________________________________________________________________________
Bot Reply: Sorry, I found personal information in your query. Please remove it first.
__________________________________________________________________________________________________________________________________________________________________________________________
__________________________________________________________________________________________________________________________________________________________________________________________
Bot Reply: I will handle as soon as possible
__________________________________________________________________________________________________________________________________________________________________________________________
_________________________________________________________________________________________________________________________

__________________________________________________________________________________________________________________________________________________________________________________________
Bot Reply: Safe reply
__________________________________________________________________________________________________________________________________________________________________________________________
